## Create Key Pair
Creates an EC2 key pair on AWS and saves the private key locally as a `.pem` file for SSH access.

In [3]:
import boto3

ec2 = boto3.client('ec2', region_name='ap-south-2')

key_pair_name = 'demo-key-value'

response = ec2.create_key_pair(KeyName=key_pair_name)

private_key = response['KeyMaterial']

# Save the private key to a .pem file (keep this secure!)
pem_file = f'{key_pair_name}.pem'
with open(pem_file, 'w') as f:
    f.write(private_key)

print(f"Key pair '{key_pair_name}' created successfully.")
print(f"Private key saved to: {pem_file}")
print(f"Key Pair ID: {response['KeyPairId']}")


Key pair 'demo-key-value' created successfully.
Private key saved to: demo-key-value.pem
Key Pair ID: key-0aaac580d242e98a1


## Delete Key Pair
Deletes the EC2 key pair from AWS and removes the local `.pem` file.

In [2]:
import os

# Delete the key pair from AWS
ec2.delete_key_pair(KeyName=key_pair_name)
print(f"Key pair '{key_pair_name}' deleted from AWS.")

# Remove the local .pem file
if os.path.exists(pem_file):
    os.remove(pem_file)
    print(f"Local file '{pem_file}' deleted.")
else:
    print(f"Local file '{pem_file}' not found.")

Key pair 'demo-key-value' deleted from AWS.
Local file 'demo-key-value.pem' deleted.


## 1. Create an EC2 Instance
Defines configuration and creates an EC2 instance using `run_instances`.

In [7]:
import boto3

ec2 = boto3.client('ec2', region_name='ap-south-2')

response = ec2.run_instances(
    ImageId='ami-011bcca8b6d63b21c',  # AMI ID (OS image) — replace with a valid AMI for your region
    InstanceType='t3.micro',          # Nitro-based instance (required for UEFI AMIs; also free-tier eligible)
    MinCount=1,                       # Minimum number of instances to launch
    MaxCount=1,                       # Maximum number of instances to launch
    KeyName='demo-key-value',         # Key pair name used for SSH access
    TagSpecifications=[{
        'ResourceType': 'instance',
        'Tags': [{'Key': 'Name', 'Value': 'MyEC2Instance'}]  # Tag to identify the instance
    }]
)

instance_id = response['Instances'][0]['InstanceId']
print(f"Instance created: {instance_id}")

Instance created: i-00952ce885aa2f757


## 2. Launch an EC2 Instance (Wait Until Running)
Waits for the instance to reach the `running` state after creation.

In [8]:
ec2_resource = boto3.resource('ec2', region_name='ap-south-2')

# Get the instance object using the instance ID from the previous cell
instance = ec2_resource.Instance(instance_id)

# Block until the instance transitions to 'running' state
instance.wait_until_running()

# Reload attributes to get the latest public IP and DNS after startup
instance.reload()

print(f"Instance is running.")
print(f"Public IP  : {instance.public_ip_address}")  # Use this IP to SSH into the instance
print(f"Public DNS : {instance.public_dns_name}")     # DNS hostname for the instance

Instance is running.
Public IP  : 16.112.58.68
Public DNS : ec2-16-112-58-68.ap-south-2.compute.amazonaws.com
